# Simulation: pass, reject, fail, time out, cancel

A launch that starts always returns a `Result`. This notebook runs a Verilator simulation and
shows each way it can end: `passed`, `rejected`, `failed` (also by a time limit) and `canceled`.
It ends with a trap: a later run can replace the files that an earlier result describes.

The notebook needs Verilator on your `PATH`. It writes only under `notebook-output/` in its
working directory.

In [ ]:
import os
from pathlib import Path

import xeda
from xeda import Design, Spec, Workspace

# The examples of the Xeda checkout. Set XEDA_EXAMPLES to use another copy.
EXAMPLES = Path(os.environ.get("XEDA_EXAMPLES", Path.cwd().parent / "examples")).resolve()
if not EXAMPLES.is_dir():
    raise FileNotFoundError(
        f"No examples directory at {EXAMPLES}. "
        "Set the XEDA_EXAMPLES environment variable to the `examples` directory of an Xeda checkout."
    )

# This notebook writes only under WORK.
WORK = Path("notebook-output").resolve()
ws = Workspace(WORK / "xeda_run")
print("Xeda", xeda.__version__)
print("run root:", ws.run_root)

from xeda import CancelToken
from xeda.errors import DesignError, RunFailedError, RunReplacedError

In [ ]:
design = Design.load(EXAMPLES / "sv/fifo/fifo.xeda.yaml")

## A run that passes

A simulation passes only on evidence that it ended: the testbench called `$finish` (or the
requested stop time or cycle count came). Xeda keeps that evidence under the metric `sim`.

In [ ]:
sim = ws.run(Spec("verilator", design))
print(sim.outcome, sim.passed)

In [ ]:
print(sim.record.metric("sim.ended_by"), sim.record.metric("sim.errors"))

## A testbench that never ends

The next cases need testbenches that misbehave on purpose. The cell below writes one file with
three testbenches and a design file for it. Both go under `WORK`.

- `tb_drained` stops without `$finish`.
- `tb_forever` runs a clock and never ends. The time limit and the cancel below always stop it.

The design file has no RTL: Verilator needs only the testbench.

In [ ]:
BENCH = WORK / "bench"
BENCH.mkdir(parents=True, exist_ok=True)
(BENCH / "bench.sv").write_text("""\
module tb_drained;
  initial #10 $display("the queue of events is empty now");
endmodule

module tb_forever;
  logic clk = 0;
  always #5 clk = ~clk;
endmodule
""")
(BENCH / "bench.xeda.yaml").write_text("""\
name: bench
tb:
  sources: [bench.sv]
  top: tb_drained
""")
bench = Design.load(BENCH / "bench.xeda.yaml")
print(bench.name, bench.tb.top)

## A run that is rejected

The tool ran and exited with status 0, but the run did not end as intended: `tb_drained` has no
event left and never calls `$finish`. A simulation that stops this way proves nothing, so Xeda
rejects the run. The outcome is `rejected` and the error code is `rejected`. The message tells you
what to do.

In [ ]:
drained = ws.run(Spec("verilator", bench))
print(drained.outcome, drained.error.code)

In [ ]:
print(drained.record.metric("sim.ended_by"))
print(drained.error.message)

A rejection is a verdict on a finished run, so Xeda stores it and reports it again unchanged when
you launch the same request again.

## A run that fails

When the tool itself fails, the outcome is `failed`. A top module that does not exist makes the
build fail. No exception is raised: read the verdict from the result.

In [ ]:
broken = design.with_updates({"tb": {"top": "no_such_module"}})
bad = ws.run(Spec("verilator", broken))
print(bad.outcome, bad.error.code)

The record of a failed run exists, because the run started. `error.log` is the key of the log to
read.

In [ ]:
print("exit status:", bad.error.exit_status)
with bad.record.open() as run:
    log = run.artifacts[bad.error.log].read_text()
print([line for line in log.splitlines() if line.startswith("%Error")])

A failed run is never reused. If you launch it again, the tool runs again.

## A time limit

The setting `timeout` is a physical quantity, so it is text with a unit. The tool runs past the
limit, Xeda stops it and the outcome is `failed` with the error code `timeout`.

In [ ]:
forever = bench.with_updates({"tb": {"top": "tb_forever"}})
slow = ws.run(Spec("verilator", forever, settings={"verilator": {"timeout": "3 s"}}))
print(slow.outcome, slow.error.code)

## A cancel

A `CancelToken` stops a launch from another thread. Here a timer cancels the launch after three
seconds. Xeda stops the tool and returns a `Result` with the outcome `canceled`. The error belongs
to the launch, so it has no node.

In [ ]:
import threading

token = CancelToken()
threading.Timer(3.0, token.cancel).start()
late = ws.run(Spec("verilator", forever), cancel=token)
print(late.outcome, late.error.code, late.error.node)

`KeyboardInterrupt` is different. It is not an outcome. It propagates after Xeda stops the tools
and releases its locks.

## Raise on a bad outcome

`require_passed` turns a bad outcome into an exception. It returns the same `Result` when the run
passed. The exception carries the result.

In [ ]:
try:
    drained.require_passed()
except RunFailedError as error:
    print(error.code, error.result.error.code)

## The trap: a later run replaces an earlier one

With the default layout, a flow of a design has one run directory. A run with other settings
replaces the files of the run before it. A record that you kept says so when you open it.

In [ ]:
first = ws.run(Spec("verilator", design))
ws.run(Spec("verilator", design, settings={"verilator": {"random_seed": 7}}))
try:
    with first.record.open():
        pass
except RunReplacedError as error:
    print(error.code)

`Workspace(..., hashed_run_dirs=True)` gives each configuration its own run directory, so both
records stay valid. A sweep over variants needs this option.

In [ ]:
kept = Workspace(WORK / "xeda_run_hashed", hashed_run_dirs=True)
first = kept.run(Spec("verilator", design))
kept.run(Spec("verilator", design, settings={"verilator": {"random_seed": 7}}))
with first.record.open() as run:
    print("still open:", run.run_dir.name.startswith("verilator_"))

## Errors that are not outcomes

A request that cannot start a run raises an exception. A design file that does not exist is one
case. The rule: if a run started, you get a `Result`; if it did not start, you get an exception
from `xeda.errors`.

A launch reads the environment of the process once, at its start. A tool inherits it, and a
changed variable can make a finished node stale.

In [ ]:
try:
    Design.load(EXAMPLES / "sv/fifo/missing.xeda.yaml")
except DesignError as error:
    print(error.code)